# 第 6 讲：聚类分析与 PCA 降维

> 用 K-means 和 PCA 训练无监督建模，输出聚类画像和降维图。

## 课件导学

**任务情境**：无监督模型用于画像、分群和结构发现，本讲训练聚类结果能不能解释。

**关键概念**

- 标准化
- K-means
- 肘部法
- PCA 可视化

**本讲产物**

- k_selection.csv
- cluster_profile.csv
- clustering_pca.png

## 资料链接与阅读抓手

下面这些链接优先选官方文档或稳定资料。课前先看标题和示例，课堂中遇到参数或概念再回查。

- [scikit-learn K-means](https://scikit-learn.org/stable/modules/clustering.html#k-means)：理解 K-means 的适用场景和参数。
- [scikit-learn PCA](https://scikit-learn.org/stable/modules/decomposition.html#pca)：学习主成分降维的解释方式。
- [scikit-learn 用户指南](https://scikit-learn.org/stable/user_guide.html)：把聚类、降维和预处理放进同一建模地图。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-06"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

## 可运行示例与讲解路线

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

n = 240
features = pd.DataFrame({
    "spending": rng.normal(500, 120, n),
    "visits": rng.normal(12, 4, n),
    "service_time": rng.normal(30, 8, n),
    "satisfaction": rng.normal(75, 10, n),
})
scaled = StandardScaler().fit_transform(features)
inertia = []
for k in range(2, 8):
    inertia.append(KMeans(n_clusters=k, random_state=42, n_init=10).fit(scaled).inertia_)
pd.DataFrame({"k": range(2, 8), "inertia": inertia}).to_csv(OUTPUT_ROOT / "k_selection.csv", index=False)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(scaled)
features["cluster"] = kmeans.labels_
profile = features.groupby("cluster").mean()
profile.to_csv(OUTPUT_ROOT / "cluster_profile.csv", encoding="utf-8-sig")
profile

In [ ]:
pca = PCA(n_components=2, random_state=42)
score = pca.fit_transform(scaled)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(2, 8), inertia, marker="o")
axes[0].set_title("Elbow method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
scatter = axes[1].scatter(score[:, 0], score[:, 1], c=features["cluster"], cmap="viridis", alpha=0.75)
axes[1].set_title("PCA view of clusters")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
fig.colorbar(scatter, ax=axes[1])
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "clustering_pca.png", dpi=160)
plt.show()
print("Explained variance:", pca.explained_variance_ratio_)

## 实战环节

**课堂任务**

- 比较 k=2、3、4 的聚类画像。
- 解释标准化前后结果为什么会变化。
- 给每一类起一个业务标签。

**课后挑战**：把 PCA 图中的聚类结果写成 150 字论文式描述。

**验收清单**

- 是否标准化
- 是否说明 K 值选择依据
- 聚类画像是否能转成实际含义

In [ ]:
lesson_resources = [{'title': 'scikit-learn K-means', 'url': 'https://scikit-learn.org/stable/modules/clustering.html#k-means', 'reading_note': '理解 K-means 的适用场景和参数。'}, {'title': 'scikit-learn PCA', 'url': 'https://scikit-learn.org/stable/modules/decomposition.html#pca', 'reading_note': '学习主成分降维的解释方式。'}, {'title': 'scikit-learn 用户指南', 'url': 'https://scikit-learn.org/stable/user_guide.html', 'reading_note': '把聚类、降维和预处理放进同一建模地图。'}]
lesson_deliverables = ['k_selection.csv', 'cluster_profile.csv', 'clustering_pca.png']
lesson_checklist = ['是否标准化', '是否说明 K 值选择依据', '聚类画像是否能转成实际含义']

pd.DataFrame(lesson_resources).to_csv(OUTPUT_ROOT / "lesson_resources.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"deliverable": lesson_deliverables}).to_csv(OUTPUT_ROOT / "lesson_deliverables.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"check_item": lesson_checklist}).to_csv(OUTPUT_ROOT / "lesson_checklist.csv", index=False, encoding="utf-8-sig")
print("Saved courseware resource map, deliverables, and checklist.")